# Chapter 16: Kalman Filter

<a href="../lite/lab/index.html?path=ch16_kalman_filter.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite: run and edit this notebook</a>

*Runs entirely in your browser with no installation required.*

**How to use:** Edit the parameter values in each cell and re-run it to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.stats import norm
from scipy.linalg import solve_discrete_are

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 11

def draw_cov_ellipse(ax, mu, cov, n_std=2, **kwargs):
    """Draw a 2D covariance ellipse at `n_std` standard deviations."""
    vals, vecs = np.linalg.eigh(cov)
    vals = np.maximum(vals, 1e-12)
    angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    w, h = 2 * n_std * np.sqrt(vals[1]), 2 * n_std * np.sqrt(vals[0])
    ell = patches.Ellipse(mu, w, h, angle=angle, **kwargs)
    ax.add_patch(ell)
    return ell

print('Imports OK.')

```{admonition} What you will build
:class: tip

- Implement a complete Kalman filter from scratch in both 1D and 2D
- Understand the Kalman gain as an optimal weighted average between prediction and measurement
- Watch covariance converge and understand why it reaches a steady state
- Track a moving robot with noisy sensors and see the filter outperform both raw measurements and dead reckoning
- Build intuition for why the Kalman filter is optimal for linear Gaussian systems

**Real world application:** The Kalman filter runs inside GPS receivers, smartphone orientation sensors, autopilots, and industrial controllers. It is the most widely deployed estimation algorithm in engineering history.
```

```{admonition} Libraries and tools used in practice
:class: note

| Library / Tool | What it does |
|---|---|
| **FilterPy** | Python Kalman filter library (KalmanFilter class) |
| **robot_localization (ROS 2)** | Production EKF/UKF for fusing IMU, odometry, GPS |
| **Eigen + custom** | Most production Kalman filters are hand-written in C++ with Eigen |
```

## 16.1 Linear Gaussian Assumption

The Kalman filter assumes three things: **linear dynamics**, **linear measurements**, and **Gaussian noise**.

**State transition** (how the robot evolves):

$$\mathbf{x}_{t} = F \,\mathbf{x}_{t-1} + B \,\mathbf{u}_{t} + \mathbf{w}, \qquad \mathbf{w} \sim \mathcal{N}(0, Q)$$

**Measurement model** (how the sensor observes the state):

$$\mathbf{z}_{t} = H \,\mathbf{x}_{t} + \mathbf{v}, \qquad \mathbf{v} \sim \mathcal{N}(0, R)$$

Here $F$ is the state transition matrix, $B$ maps control inputs, $H$ selects which states the sensor sees, $Q$ is the **process noise covariance**, and $R$ is the **measurement noise covariance**.

The Kalman filter's job: combine noisy motion predictions and noisy sensor readings into the **best possible estimate** of the true state.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
true_velocity     = 1.0    # m/step  (try 0.5, 1.0, 2.0)
sigma_motion      = 0.4    # process noise std   (try 0.1, 0.4, 1.0)
sigma_sensor      = 1.0    # measurement noise   (try 0.3, 1.0, 3.0)
n_steps           = 50     # number of time steps (try 20, 50, 100)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# Ground truth: robot moves right with constant velocity + small process noise
true_pos = np.zeros(n_steps)
for t in range(1, n_steps):
    true_pos[t] = true_pos[t-1] + true_velocity + np.random.normal(0, sigma_motion)

# Dead reckoning: integrate noisy velocity commands (no sensor correction)
dead_reckoning = np.zeros(n_steps)
for t in range(1, n_steps):
    dead_reckoning[t] = dead_reckoning[t-1] + true_velocity + np.random.normal(0, sigma_motion)

# Noisy measurements of true position
measurements = true_pos + np.random.normal(0, sigma_sensor, n_steps)

steps = np.arange(n_steps)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(steps, true_pos, 'k-', lw=2.5, label='True position', zorder=5)
ax.plot(steps, dead_reckoning, color='orange', lw=1.5, alpha=0.8,
        label='Dead reckoning (odometry only)', zorder=3)
ax.scatter(steps, measurements, s=18, color='tomato', alpha=0.5,
           label='Noisy measurements', zorder=2)
ax.set_xlabel('Time step'); ax.set_ylabel('Position (m)')
ax.set_title('The Estimation Problem: True State, Noisy Odometry, Noisy Sensor')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print('Dead reckoning drifts from truth over time.')
print('Measurements are scattered around truth.')
print('The Kalman filter will combine both to produce the best estimate.')

## 16.2 Prediction Step

The **predict** step propagates the belief forward using the motion model:

$$\hat{\mathbf{x}}_{t|t-1} = F\,\hat{\mathbf{x}}_{t-1|t-1} + B\,\mathbf{u}_t$$

$$P_{t|t-1} = F\,P_{t-1|t-1}\,F^T + Q$$

The mean follows the motion model, and the covariance **always grows**. Prediction without measurement means losing confidence over time.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
initial_mean     = 0.0    # starting position
initial_var      = 0.5    # initial uncertainty
velocity_cmd     = 1.0    # commanded velocity per step
process_var      = 0.3    # Q: process noise variance (try 0.1, 0.3, 1.0)
n_predict_steps  = 10     # number of prediction-only steps
# ─────────────────────────────────────────────────────────────────────────────

mu = initial_mean
var = initial_var

means = [mu]
variances_pred = [var]

for t in range(n_predict_steps):
    mu = mu + velocity_cmd          # F=1, B=1, u=velocity_cmd
    var = var + process_var          # P = F*P*F' + Q  (scalars: F=1)
    means.append(mu)
    variances_pred.append(var)

means = np.array(means)
stds = np.sqrt(np.array(variances_pred))
ts = np.arange(len(means))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: belief Gaussians getting wider
x_range = np.linspace(means[0] - 4*stds[-1], means[-1] + 4*stds[-1], 500)
colors = plt.cm.Blues(np.linspace(0.3, 1.0, len(means)))
for i in range(len(means)):
    pdf = norm.pdf(x_range, means[i], stds[i])
    ax1.fill_between(x_range, pdf, alpha=0.15, color=colors[i])
    ax1.plot(x_range, pdf, color=colors[i], lw=1.5, label=f't={i}' if i % 2 == 0 else '')
ax1.set_xlabel('Position (m)'); ax1.set_ylabel('Probability density')
ax1.set_title('Belief After Each Prediction Step')
ax1.legend(fontsize=8, ncol=2)

# Right: variance growing linearly
ax2.plot(ts, variances_pred, 'o-', color='steelblue', lw=2, markersize=6)
ax2.set_xlabel('Time step'); ax2.set_ylabel('Variance $\\sigma^2$')
ax2.set_title('Covariance Grows Without Measurements')
ax2.axhline(initial_var, ls='--', color='gray', alpha=0.5, label='Initial variance')
ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'After {n_predict_steps} predictions: mean = {means[-1]:.1f}, variance = {variances_pred[-1]:.2f}')
print('Key observation: without measurements, uncertainty grows without bound.')

## 16.3 Update Step

The **update** step incorporates a measurement to sharpen the estimate:

$$K = P_{t|t-1}\,H^T\,(H\,P_{t|t-1}\,H^T + R)^{-1}$$

$$\hat{\mathbf{x}}_{t|t} = \hat{\mathbf{x}}_{t|t-1} + K\,(\mathbf{z}_t - H\,\hat{\mathbf{x}}_{t|t-1})$$

$$P_{t|t} = (I - K\,H)\,P_{t|t-1}$$

The term $(\mathbf{z}_t - H\,\hat{\mathbf{x}}_{t|t-1})$ is the **innovation**: how surprised we are by the measurement. $K$ scales this surprise. The update **always reduces** uncertainty.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
prior_mean   = 5.0     # predicted state mean
prior_var    = 2.0     # predicted state variance  (try 0.5, 2.0, 5.0)
meas_value   = 7.0     # measurement value         (try 4.0, 7.0, 10.0)
meas_var     = 1.0     # measurement variance R    (try 0.3, 1.0, 4.0)
# ─────────────────────────────────────────────────────────────────────────────

# Kalman update (1D)
K = prior_var / (prior_var + meas_var)
post_mean = prior_mean + K * (meas_value - prior_mean)
post_var = (1 - K) * prior_var

x_range = np.linspace(
    min(prior_mean, meas_value) - 4*np.sqrt(max(prior_var, meas_var)),
    max(prior_mean, meas_value) + 4*np.sqrt(max(prior_var, meas_var)),
    500
)

prior_pdf = norm.pdf(x_range, prior_mean, np.sqrt(prior_var))
meas_pdf  = norm.pdf(x_range, meas_value, np.sqrt(meas_var))
post_pdf  = norm.pdf(x_range, post_mean, np.sqrt(post_var))

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(x_range, prior_pdf, alpha=0.2, color='steelblue')
ax.plot(x_range, prior_pdf, 'steelblue', lw=2, label=f'Prior (prediction): $\\mu$={prior_mean}, $\\sigma^2$={prior_var}')
ax.fill_between(x_range, meas_pdf, alpha=0.2, color='tomato')
ax.plot(x_range, meas_pdf, 'tomato', lw=2, label=f'Likelihood (measurement): z={meas_value}, R={meas_var}')
ax.fill_between(x_range, post_pdf, alpha=0.3, color='forestgreen')
ax.plot(x_range, post_pdf, 'forestgreen', lw=2.5,
        label=f'Posterior (updated): $\\mu$={post_mean:.2f}, $\\sigma^2$={post_var:.2f}')
ax.axvline(prior_mean, color='steelblue', ls=':', alpha=0.6)
ax.axvline(meas_value, color='tomato', ls=':', alpha=0.6)
ax.axvline(post_mean, color='forestgreen', ls=':', alpha=0.8)
ax.set_xlabel('State value'); ax.set_ylabel('Probability density')
ax.set_title(f'Gaussian Product: Prior $\\times$ Likelihood = Posterior   (K = {K:.3f})')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print(f'Kalman gain K = {K:.3f}')
print(f'The posterior is always narrower than both the prior and the likelihood.')
print(f'The posterior mean sits between the prior mean and the measurement,')
print(f'weighted by their relative certainties.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
prior_mean_base  = 5.0       # predicted state mean
prior_var_base   = 2.5       # predicted state variance
meas_var_base    = 1.0       # measurement noise variance
meas_locations   = [3.0, 6.0, 9.0]  # three different measurement values
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
x_range = np.linspace(-1, 14, 500)

for ax, z in zip(axes, meas_locations):
    K = prior_var_base / (prior_var_base + meas_var_base)
    post_mu = prior_mean_base + K * (z - prior_mean_base)
    post_var_v = (1 - K) * prior_var_base

    prior_pdf = norm.pdf(x_range, prior_mean_base, np.sqrt(prior_var_base))
    meas_pdf  = norm.pdf(x_range, z, np.sqrt(meas_var_base))
    post_pdf  = norm.pdf(x_range, post_mu, np.sqrt(post_var_v))

    ax.fill_between(x_range, prior_pdf, alpha=0.15, color='steelblue')
    ax.plot(x_range, prior_pdf, 'steelblue', lw=1.5, label='Prior')
    ax.fill_between(x_range, meas_pdf, alpha=0.15, color='tomato')
    ax.plot(x_range, meas_pdf, 'tomato', lw=1.5, label='Measurement')
    ax.fill_between(x_range, post_pdf, alpha=0.25, color='forestgreen')
    ax.plot(x_range, post_pdf, 'forestgreen', lw=2, label='Posterior')
    ax.axvline(post_mu, color='forestgreen', ls=':', lw=1.5)
    ax.set_title(f'z = {z:.1f} → posterior at {post_mu:.2f}', fontsize=10)
    ax.set_xlabel('State value')
    ax.legend(fontsize=7)

axes[0].set_ylabel('Density')
fig.suptitle('Update Step with Three Different Measurements', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

print('The posterior always sits between the prior and the measurement.')
print('Closer to whichever has smaller variance (higher confidence).')

## 16.4 Kalman Gain Intuition

The **Kalman gain** $K$ is the heart of the filter. In 1D it simplifies to:

$$K = \frac{P}{P + R}$$

This is exactly the **inverse variance weighting** from Chapter 5:

- If the prediction is uncertain ($P$ large) then $K \approx 1$ and the filter trusts the **measurement**.
- If the sensor is noisy ($R$ large) then $K \approx 0$ and the filter trusts the **prediction**.
- When $P = R$, the gain is $K = 0.5$: equal trust in both.

**Key fact:** No other linear combination of prediction and measurement gives a lower expected error. The Kalman gain is **optimal**.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
pred_mean   = 5.0     # predicted state
pred_var    = 2.0     # prediction variance P
z_value     = 8.0     # measurement value
R_values    = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]   # sweep R
# ─────────────────────────────────────────────────────────────────────────────

K_list = []
post_means = []
post_vars_list = []

for R_val in R_values:
    K = pred_var / (pred_var + R_val)
    pm = pred_mean + K * (z_value - pred_mean)
    pv = (1 - K) * pred_var
    K_list.append(K)
    post_means.append(pm)
    post_vars_list.append(pv)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: K vs R
ax1.plot(R_values, K_list, 'o-', color='steelblue', lw=2, markersize=8)
ax1.axhline(0.5, ls='--', color='gray', alpha=0.5, label='K = 0.5 (equal trust)')
ax1.set_xlabel('Measurement noise $R$'); ax1.set_ylabel('Kalman gain $K$')
ax1.set_title(f'Kalman Gain vs. Measurement Noise (P = {pred_var})')
ax1.legend(fontsize=9)
for r, k in zip(R_values, K_list):
    ax1.annotate(f'{k:.2f}', (r, k), textcoords='offset points',
                 xytext=(5, 8), fontsize=8, color='steelblue')

# Right: updated mean vs R
ax2.plot(R_values, post_means, 'o-', color='forestgreen', lw=2, markersize=8)
ax2.axhline(pred_mean, ls='--', color='steelblue', alpha=0.7, label=f'Prediction = {pred_mean}')
ax2.axhline(z_value, ls='--', color='tomato', alpha=0.7, label=f'Measurement = {z_value}')
ax2.set_xlabel('Measurement noise $R$'); ax2.set_ylabel('Updated estimate')
ax2.set_title('Updated Mean Moves Between Prediction and Measurement')
ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()

print('Small R (precise sensor):  K is near 1, trust measurement')
print('Large R (noisy sensor):    K is near 0, trust prediction')
print('This is optimal inverse-variance weighting.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
pred_mu_demo   = 4.0
pred_var_demo  = 3.0
z_demo         = 7.0
R_scenarios    = [0.3, 3.0, 30.0]   # precise, balanced, noisy
scenario_names = ['Precise sensor (R=0.3)', 'Balanced (R=3.0)', 'Noisy sensor (R=30.0)']
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
x_range = np.linspace(-2, 14, 500)

for ax, R_val, name in zip(axes, R_scenarios, scenario_names):
    K = pred_var_demo / (pred_var_demo + R_val)
    pm = pred_mu_demo + K * (z_demo - pred_mu_demo)
    pv = (1 - K) * pred_var_demo

    prior_pdf = norm.pdf(x_range, pred_mu_demo, np.sqrt(pred_var_demo))
    meas_pdf  = norm.pdf(x_range, z_demo, np.sqrt(R_val))
    post_pdf  = norm.pdf(x_range, pm, np.sqrt(pv))

    ax.plot(x_range, prior_pdf, 'steelblue', lw=2, label='Prediction')
    ax.plot(x_range, meas_pdf, 'tomato', lw=2, label='Measurement')
    ax.plot(x_range, post_pdf, 'forestgreen', lw=2.5, label='Updated')
    ax.axvline(pm, color='forestgreen', ls=':', alpha=0.8)
    ax.set_title(f'{name}\nK = {K:.3f}', fontsize=10)
    ax.set_xlabel('State value')
    ax.legend(fontsize=7)

axes[0].set_ylabel('Density')
fig.suptitle('Three Scenarios: How K Shifts the Estimate', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Inverse-variance weighting connection ────────────────────────────────────
# The Kalman update is equivalent to the inverse-variance weighted average
# from Chapter 5:
#   x_fused = (x_pred / P + z / R) / (1/P + 1/R)
#   P_fused = 1 / (1/P + 1/R)

P_ex = 2.5
R_ex = 1.0
x_pred_ex = 4.0
z_ex = 7.0

# Kalman form
K_ex = P_ex / (P_ex + R_ex)
x_kalman = x_pred_ex + K_ex * (z_ex - x_pred_ex)
P_kalman = (1 - K_ex) * P_ex

# Inverse-variance form
x_iv = (x_pred_ex / P_ex + z_ex / R_ex) / (1/P_ex + 1/R_ex)
P_iv = 1.0 / (1/P_ex + 1/R_ex)

print('Kalman form:')
print(f'  x_updated = {x_kalman:.6f},  P_updated = {P_kalman:.6f}')
print(f'Inverse-variance form:')
print(f'  x_updated = {x_iv:.6f},  P_updated = {P_iv:.6f}')
print(f'\nThey are identical: difference = {abs(x_kalman - x_iv):.2e}')
print('The Kalman update IS inverse-variance weighting.')

## 16.5 Covariance Behavior

Each **prediction** grows the covariance by $Q$. Each **update** shrinks it. Over time, these two effects balance out and the covariance **converges to a steady state**.

The steady state covariance $P_\infty$ satisfies the **Discrete Algebraic Riccati Equation** (DARE):

$$P_\infty = F\,P_\infty\,F^T + Q - F\,P_\infty\,H^T\,(H\,P_\infty\,H^T + R)^{-1}\,H\,P_\infty\,F^T$$

In 1D with $F=H=1$, the steady state simplifies to:

$$P_\infty = \frac{-Q + \sqrt{Q^2 + 4QR}}{2}$$

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
Q_cov      = 0.3     # process noise variance  (try 0.1, 0.3, 1.0)
R_cov      = 1.0     # measurement noise variance (try 0.5, 1.0, 3.0)
P_init     = 10.0    # initial covariance (try 0.1, 1.0, 10.0, 100.0)
n_cov      = 100     # number of steps
# ─────────────────────────────────────────────────────────────────────────────

P = P_init
P_history = [P]
K_history = []

for t in range(n_cov):
    # Predict
    P_pred = P + Q_cov
    # Update
    K = P_pred / (P_pred + R_cov)
    P = (1 - K) * P_pred
    P_history.append(P)
    K_history.append(K)

# Analytical steady state (1D, F=H=1)
P_ss_analytic = (-Q_cov + np.sqrt(Q_cov**2 + 4*Q_cov*R_cov)) / 2

# DARE solution via scipy
F_m = np.array([[1.0]])
H_m = np.array([[1.0]])
Q_m = np.array([[Q_cov]])
R_m = np.array([[R_cov]])
P_dare = solve_discrete_are(F_m.T, H_m.T, Q_m, R_m)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(P_history, 'steelblue', lw=2, label='Filter covariance $P$')
ax1.axhline(P_ss_analytic, ls='--', color='forestgreen', lw=2,
            label=f'Steady state (analytic) = {P_ss_analytic:.4f}')
ax1.axhline(P_dare[0, 0], ls=':', color='orange', lw=2,
            label=f'Steady state (DARE) = {P_dare[0,0]:.4f}')
ax1.set_xlabel('Time step'); ax1.set_ylabel('Covariance $P$')
ax1.set_title('Covariance Convergence to Steady State')
ax1.legend(fontsize=8)

ax2.plot(K_history, 'tomato', lw=2, label='Kalman gain $K$')
K_ss = P_ss_analytic / (P_ss_analytic + R_cov)
ax2.axhline(K_ss, ls='--', color='forestgreen', lw=2,
            label=f'Steady state K = {K_ss:.4f}')
ax2.set_xlabel('Time step'); ax2.set_ylabel('Kalman gain $K$')
ax2.set_title('Kalman Gain Convergence')
ax2.legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f'P converges to {P_history[-1]:.6f} (analytic: {P_ss_analytic:.6f})')
print(f'K converges to {K_history[-1]:.6f}')
print('After convergence, the filter has reached its optimal operating point.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
Q_sweep = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]   # process noise values
R_sweep = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]   # measurement noise values
# ─────────────────────────────────────────────────────────────────────────────

P_ss_grid = np.zeros((len(Q_sweep), len(R_sweep)))
for i, q in enumerate(Q_sweep):
    for j, r in enumerate(R_sweep):
        P_ss_grid[i, j] = (-q + np.sqrt(q**2 + 4*q*r)) / 2

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(P_ss_grid, origin='lower', aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(R_sweep))); ax.set_xticklabels([f'{r}' for r in R_sweep])
ax.set_yticks(range(len(Q_sweep))); ax.set_yticklabels([f'{q}' for q in Q_sweep])
ax.set_xlabel('Measurement noise $R$'); ax.set_ylabel('Process noise $Q$')
ax.set_title('Steady State Covariance $P_\\infty$ vs. $Q$ and $R$')

# Annotate each cell
for i in range(len(Q_sweep)):
    for j in range(len(R_sweep)):
        ax.text(j, i, f'{P_ss_grid[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='black' if P_ss_grid[i,j] < 2 else 'white')

plt.colorbar(im, label='$P_\\infty$'); plt.tight_layout(); plt.show()

print('Large Q (noisy motion) + large R (noisy sensor) = large steady state uncertainty.')
print('Small Q (precise motion) + small R (precise sensor) = small steady state uncertainty.')
print('The ratio Q/R determines how much the filter trusts motion vs. measurement.')

## 16.6 Complete 1D Example

A robot drives along a line. It receives velocity commands corrupted by noise, and a sensor measures its position with noise. We implement the complete **predict/update** loop and compare the Kalman filter to dead reckoning and raw measurements.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
sigma_motion_1d  = 0.5    # process noise std      (try 0.1, 0.5, 1.5)
sigma_sensor_1d  = 1.5    # measurement noise std  (try 0.3, 1.5, 4.0)
true_vel_1d      = 1.0    # true velocity per step (try 0.5, 1.0, 2.0)
n_steps_1d       = 80     # number of time steps   (try 30, 80, 200)
seed_1d          = 42
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(seed_1d)

Q_1d = sigma_motion_1d**2
R_1d = sigma_sensor_1d**2

# Ground truth
true_pos_1d = np.zeros(n_steps_1d)
for t in range(1, n_steps_1d):
    true_pos_1d[t] = true_pos_1d[t-1] + true_vel_1d + np.random.normal(0, sigma_motion_1d)

# Dead reckoning (integrating noisy velocity, no sensor)
dead_reck = np.zeros(n_steps_1d)
for t in range(1, n_steps_1d):
    dead_reck[t] = dead_reck[t-1] + true_vel_1d + np.random.normal(0, sigma_motion_1d)

# Noisy measurements
meas_1d = true_pos_1d + np.random.normal(0, sigma_sensor_1d, n_steps_1d)

# Kalman filter
mu_kf = 0.0
P_kf = 1.0
est_kf = np.zeros(n_steps_1d)
P_kf_hist = np.zeros(n_steps_1d)
K_kf_hist = np.zeros(n_steps_1d)

est_kf[0] = mu_kf
P_kf_hist[0] = P_kf

for t in range(1, n_steps_1d):
    # Predict
    mu_kf = mu_kf + true_vel_1d     # F=1, B=1, u=vel
    P_kf = P_kf + Q_1d
    # Update
    K = P_kf / (P_kf + R_1d)
    mu_kf = mu_kf + K * (meas_1d[t] - mu_kf)
    P_kf = (1 - K) * P_kf
    est_kf[t] = mu_kf
    P_kf_hist[t] = P_kf
    K_kf_hist[t] = K

steps_1d = np.arange(n_steps_1d)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Top: trajectories
ax1.plot(steps_1d, true_pos_1d, 'k-', lw=2.5, label='True position', zorder=5)
ax1.plot(steps_1d, dead_reck, color='orange', lw=1.5, alpha=0.7,
         label='Dead reckoning', zorder=3)
ax1.scatter(steps_1d, meas_1d, s=12, color='tomato', alpha=0.4,
            label='Measurements', zorder=2)
ax1.plot(steps_1d, est_kf, 'steelblue', lw=2, label='Kalman filter', zorder=4)
ax1.fill_between(steps_1d, est_kf - 2*np.sqrt(P_kf_hist),
                 est_kf + 2*np.sqrt(P_kf_hist),
                 alpha=0.15, color='steelblue', label='$\\pm 2\\sigma$')
ax1.set_ylabel('Position (m)')
ax1.set_title('Complete 1D Kalman Filter: Four Sources Compared')
ax1.legend(fontsize=8, ncol=3)

# Bottom: errors
err_dr = np.abs(dead_reck - true_pos_1d)
err_meas = np.abs(meas_1d - true_pos_1d)
err_kf = np.abs(est_kf - true_pos_1d)

ax2.plot(steps_1d, err_dr, color='orange', lw=1.5, alpha=0.7, label='Dead reckoning error')
ax2.plot(steps_1d, err_meas, color='tomato', lw=1, alpha=0.5, label='Measurement error')
ax2.plot(steps_1d, err_kf, 'steelblue', lw=2, label='KF error')
ax2.set_xlabel('Time step'); ax2.set_ylabel('Absolute error (m)')
ax2.set_title('Estimation Error Over Time')
ax2.legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f'RMS errors:')
print(f'  Dead reckoning: {np.sqrt(np.mean(err_dr**2)):.3f} m')
print(f'  Raw measurements: {np.sqrt(np.mean(err_meas**2)):.3f} m')
print(f'  Kalman filter:    {np.sqrt(np.mean(err_kf**2)):.3f} m')
print(f'\nThe Kalman filter achieves lower error than either source alone.')

In [ ]:
# ── Error statistics bar chart ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

labels = ['Dead\nreckoning', 'Raw\nmeasurements', 'Kalman\nfilter']
rmse_vals = [
    np.sqrt(np.mean(err_dr**2)),
    np.sqrt(np.mean(err_meas**2)),
    np.sqrt(np.mean(err_kf**2)),
]
colors_bar = ['orange', 'tomato', 'steelblue']

bars = ax.bar(labels, rmse_vals, color=colors_bar, edgecolor='white', width=0.5)
for bar, val in zip(bars, rmse_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylabel('RMS Error (m)'); ax.set_title('RMS Estimation Error Comparison')
plt.tight_layout(); plt.show()

## 16.7 Complete 2D Example

A robot moves in 2D. The state is $\mathbf{x} = [x, y, v_x, v_y]^T$, combining position and velocity. The sensor only observes position $[x, y]^T$ with noise. The filter must **infer velocity** from noisy position readings alone.

In [ ]:
class KalmanFilter2D:
    """2D Kalman filter: state = [x, y, vx, vy], observation = [x, y]."""

    def __init__(self, dt, sigma_accel, sigma_meas):
        self.dt = dt
        # State transition: constant velocity model
        self.F = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1,  0],
            [0, 0, 0,  1]
        ])
        # Observation: measure position only
        self.H = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ])
        # Process noise: acceleration affects velocity, which affects position
        G = np.array([[0.5*dt**2, 0],
                      [0, 0.5*dt**2],
                      [dt, 0],
                      [0, dt]])
        self.Q = (sigma_accel**2) * (G @ G.T)
        # Measurement noise
        self.R = (sigma_meas**2) * np.eye(2)
        # Initial state and covariance
        self.x = np.zeros(4)
        self.P = np.eye(4) * 10.0

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q

    def update(self, z):
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        innovation = z - self.H @ self.x
        self.x = self.x + K @ innovation
        I_KH = np.eye(4) - K @ self.H
        # Joseph form for numerical stability
        self.P = I_KH @ self.P @ I_KH.T + K @ self.R @ K.T

    @property
    def position(self):
        return self.x[:2]

    @property
    def velocity(self):
        return self.x[2:]

    @property
    def position_cov(self):
        return self.P[:2, :2]

print('KalmanFilter2D class defined.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
dt_2d         = 0.1     # time step (s)
sigma_accel   = 0.5     # process noise: acceleration std (try 0.1, 0.5, 2.0)
sigma_meas_2d = 1.0     # measurement noise std           (try 0.3, 1.0, 3.0)
n_steps_2d    = 100     # trajectory length                (try 50, 100, 200)
traj_type     = 'circle'  # 'circle' or 'figure8'
seed_2d       = 12
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(seed_2d)

t_arr = np.arange(n_steps_2d) * dt_2d

if traj_type == 'circle':
    radius = 10.0
    omega = 2 * np.pi / (n_steps_2d * dt_2d)  # one full revolution
    true_x = radius * np.cos(omega * t_arr)
    true_y = radius * np.sin(omega * t_arr)
    true_vx = -radius * omega * np.sin(omega * t_arr)
    true_vy =  radius * omega * np.cos(omega * t_arr)
elif traj_type == 'figure8':
    true_x = 10 * np.sin(2 * np.pi * t_arr / (n_steps_2d * dt_2d))
    true_y = 5 * np.sin(4 * np.pi * t_arr / (n_steps_2d * dt_2d))
    true_vx = np.gradient(true_x, dt_2d)
    true_vy = np.gradient(true_y, dt_2d)

# Noisy measurements
meas_x = true_x + np.random.normal(0, sigma_meas_2d, n_steps_2d)
meas_y = true_y + np.random.normal(0, sigma_meas_2d, n_steps_2d)

# Run filter
kf = KalmanFilter2D(dt_2d, sigma_accel, sigma_meas_2d)
kf.x = np.array([meas_x[0], meas_y[0], 0, 0])  # initialize from first measurement

est_positions = []
est_velocities = []
est_covs = []

for t in range(n_steps_2d):
    kf.predict()
    kf.update(np.array([meas_x[t], meas_y[t]]))
    est_positions.append(kf.position.copy())
    est_velocities.append(kf.velocity.copy())
    est_covs.append(kf.position_cov.copy())

est_positions = np.array(est_positions)
est_velocities = np.array(est_velocities)

# Plot 2D trajectory
fig, ax = plt.subplots(figsize=(9, 8))
ax.plot(true_x, true_y, 'k-', lw=2.5, label='True trajectory', zorder=5)
ax.scatter(meas_x, meas_y, s=12, color='tomato', alpha=0.35,
           label='Measurements', zorder=2)
ax.plot(est_positions[:, 0], est_positions[:, 1], 'steelblue', lw=2,
        label='KF estimate', zorder=4)

# Draw covariance ellipses every 5th step
for i in range(0, n_steps_2d, 5):
    draw_cov_ellipse(ax, est_positions[i], est_covs[i], n_std=2,
                     facecolor='steelblue', edgecolor='steelblue', alpha=0.1)

ax.plot(est_positions[0, 0], est_positions[0, 1], 'o', color='forestgreen',
        markersize=10, zorder=6, label='Start')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('2D Kalman Filter Tracking (ellipses = 2$\\sigma$ uncertainty)')
ax.set_aspect('equal'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

pos_err = np.sqrt((est_positions[:,0] - true_x)**2 + (est_positions[:,1] - true_y)**2)
meas_err = np.sqrt((meas_x - true_x)**2 + (meas_y - true_y)**2)
print(f'RMS position error (measurements): {np.sqrt(np.mean(meas_err**2)):.3f} m')
print(f'RMS position error (KF estimate):  {np.sqrt(np.mean(pos_err**2)):.3f} m')

In [ ]:
# ── Velocity estimation: the filter infers velocity from position data ─────

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.plot(t_arr, true_vx, 'k-', lw=2, label='True $v_x$', zorder=3)
ax1.plot(t_arr, est_velocities[:, 0], 'steelblue', lw=2, label='KF estimated $v_x$', zorder=4)
ax1.set_ylabel('Velocity $v_x$ (m/s)')
ax1.set_title('Velocity Estimates (Not Directly Measured!)')
ax1.legend(fontsize=9)

ax2.plot(t_arr, true_vy, 'k-', lw=2, label='True $v_y$', zorder=3)
ax2.plot(t_arr, est_velocities[:, 1], 'tomato', lw=2, label='KF estimated $v_y$', zorder=4)
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Velocity $v_y$ (m/s)')
ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()

vel_err = np.sqrt((est_velocities[:,0] - true_vx)**2 + (est_velocities[:,1] - true_vy)**2)
print(f'RMS velocity error: {np.sqrt(np.mean(vel_err**2)):.3f} m/s')
print('The filter estimates velocity purely from noisy position observations.')
print('This is the power of the state space formulation: unobserved states')
print('are inferred through the dynamics model.')

In [ ]:
# ── Covariance ellipse evolution ─────────────────────────────────────────────
# Show how ellipses shrink over the first 20 steps then stabilize

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
snapshot_steps = [0, 5, 15, 50]
titles = ['t = 0 (initial)', 't = 5', 't = 15', 't = 50 (steady state)']

for ax, t_idx, title in zip(axes, snapshot_steps, titles):
    t_idx = min(t_idx, len(est_covs) - 1)
    cov = est_covs[t_idx]

    draw_cov_ellipse(ax, [0, 0], cov, n_std=1, facecolor='steelblue', alpha=0.3,
                     edgecolor='steelblue', lw=2, label='1$\\sigma$')
    draw_cov_ellipse(ax, [0, 0], cov, n_std=2, facecolor='steelblue', alpha=0.1,
                     edgecolor='steelblue', lw=1, ls='--', label='2$\\sigma$')
    ax.plot(0, 0, '+', color='k', markersize=10, mew=2)
    vals = np.sqrt(np.linalg.eigvalsh(cov))
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.set_title(f'{title}\n$\\sigma_1$={vals[0]:.3f}, $\\sigma_2$={vals[1]:.3f}', fontsize=9)
    ax.legend(fontsize=7)

fig.suptitle('Position Covariance Ellipses at Different Time Steps', fontsize=12, y=1.03)
plt.tight_layout(); plt.show()

print('The ellipses shrink rapidly at first, then converge to a steady size.')

## 16.8 When the Kalman Filter Fails

The Kalman filter is optimal for **linear Gaussian** systems. When the assumptions break, performance degrades. Two common failure modes:

1. **Nonlinear dynamics:** If the robot's motion is nonlinear (turns, rotations), the linear prediction $F\mathbf{x}$ misses the true trajectory.
2. **Wrong noise parameters:** If $Q$ or $R$ are tuned incorrectly, the filter becomes either overconfident or too reactive.

These failures motivate the Extended Kalman Filter (EKF) and Unscented Kalman Filter (UKF) in Chapter 17.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
sigma_accel_fail  = 0.5
sigma_meas_fail   = 1.0
n_steps_fail      = 120
dt_fail           = 0.1
turn_start        = 40     # step when the sharp turn begins
turn_end          = 80     # step when the sharp turn ends
turn_rate         = 3.0    # how sharp the turn is (rad/s, try 1.0, 3.0, 5.0)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(99)

# Generate a trajectory with a sharp turn
true_x_f = np.zeros(n_steps_fail)
true_y_f = np.zeros(n_steps_fail)
true_vx_f = np.zeros(n_steps_fail)
true_vy_f = np.zeros(n_steps_fail)
heading = 0.0
speed = 5.0

for t in range(n_steps_fail):
    if turn_start <= t < turn_end:
        heading += turn_rate * dt_fail
    true_vx_f[t] = speed * np.cos(heading)
    true_vy_f[t] = speed * np.sin(heading)
    if t > 0:
        true_x_f[t] = true_x_f[t-1] + true_vx_f[t] * dt_fail
        true_y_f[t] = true_y_f[t-1] + true_vy_f[t] * dt_fail

meas_x_f = true_x_f + np.random.normal(0, sigma_meas_fail, n_steps_fail)
meas_y_f = true_y_f + np.random.normal(0, sigma_meas_fail, n_steps_fail)

# Run constant-velocity KF
kf_f = KalmanFilter2D(dt_fail, sigma_accel_fail, sigma_meas_fail)
kf_f.x = np.array([meas_x_f[0], meas_y_f[0], speed, 0])

est_f = []
for t in range(n_steps_fail):
    kf_f.predict()
    kf_f.update(np.array([meas_x_f[t], meas_y_f[t]]))
    est_f.append(kf_f.position.copy())
est_f = np.array(est_f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: trajectory
ax1.plot(true_x_f, true_y_f, 'k-', lw=2.5, label='True trajectory', zorder=5)
ax1.scatter(meas_x_f, meas_y_f, s=10, color='tomato', alpha=0.3, label='Measurements', zorder=2)
ax1.plot(est_f[:, 0], est_f[:, 1], 'steelblue', lw=2, label='KF estimate', zorder=4)
ax1.plot(true_x_f[turn_start], true_y_f[turn_start], 's', color='orange',
         markersize=12, zorder=6, label='Turn begins')
ax1.set_xlabel('x (m)'); ax1.set_ylabel('y (m)')
ax1.set_aspect('equal')
ax1.set_title('Constant Velocity KF Fails During Sharp Turn')
ax1.legend(fontsize=8)

# Right: error over time
err_f = np.sqrt((est_f[:,0] - true_x_f)**2 + (est_f[:,1] - true_y_f)**2)
t_arr_f = np.arange(n_steps_fail)
ax2.plot(t_arr_f, err_f, 'steelblue', lw=2)
ax2.axvspan(turn_start, turn_end, alpha=0.15, color='tomato', label='Turn region')
ax2.set_xlabel('Time step'); ax2.set_ylabel('Position error (m)')
ax2.set_title('Error Spikes During the Nonlinear Maneuver')
ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()

print('The constant velocity model predicts straight ahead while the robot turns.')
print('The error spikes during the turn, then recovers as measurements pull it back.')
print('This motivates nonlinear filters (EKF, UKF) in Chapter 17.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
# Wrong noise tuning: see what happens when Q is set incorrectly
true_Q_std    = 0.5     # actual process noise std
true_R_std    = 1.0     # actual measurement noise std
Q_too_small   = 0.01    # filter thinks motion is nearly perfect
Q_too_large   = 5.0     # filter thinks motion is very noisy
Q_correct     = 0.5     # correct value
n_steps_tune  = 80
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(77)

# Generate ground truth (1D for clarity)
true_pos_t = np.zeros(n_steps_tune)
for t in range(1, n_steps_tune):
    true_pos_t[t] = true_pos_t[t-1] + 1.0 + np.random.normal(0, true_Q_std)
meas_t = true_pos_t + np.random.normal(0, true_R_std, n_steps_tune)

def run_1d_kf(measurements_in, Q_val, R_val):
    mu, P = 0.0, 1.0
    ests, Ps = [], []
    for t in range(len(measurements_in)):
        mu = mu + 1.0
        P = P + Q_val
        K = P / (P + R_val)
        mu = mu + K * (measurements_in[t] - mu)
        P = (1 - K) * P
        ests.append(mu)
        Ps.append(P)
    return np.array(ests), np.array(Ps)

R_val = true_R_std**2
est_small, P_small = run_1d_kf(meas_t, Q_too_small**2, R_val)
est_large, P_large = run_1d_kf(meas_t, Q_too_large**2, R_val)
est_correct, P_correct = run_1d_kf(meas_t, Q_correct**2, R_val)

steps_t = np.arange(n_steps_tune)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

ax1.plot(steps_t, true_pos_t, 'k-', lw=2, label='True', zorder=5)
ax1.scatter(steps_t, meas_t, s=10, color='gray', alpha=0.3, zorder=1)
ax1.plot(steps_t, est_small, color='tomato', lw=1.5,
         label=f'Q too small ({Q_too_small}): overconfident', zorder=3)
ax1.plot(steps_t, est_large, color='orange', lw=1.5,
         label=f'Q too large ({Q_too_large}): too reactive', zorder=3)
ax1.plot(steps_t, est_correct, color='steelblue', lw=2,
         label=f'Q correct ({Q_correct})', zorder=4)
ax1.set_ylabel('Position (m)')
ax1.set_title('Effect of Wrong Process Noise Q')
ax1.legend(fontsize=8)

ax2.plot(steps_t, np.abs(est_small - true_pos_t), color='tomato', lw=1.5,
         label=f'Q={Q_too_small} error')
ax2.plot(steps_t, np.abs(est_large - true_pos_t), color='orange', lw=1.5,
         label=f'Q={Q_too_large} error')
ax2.plot(steps_t, np.abs(est_correct - true_pos_t), color='steelblue', lw=2,
         label=f'Q={Q_correct} error')
ax2.set_xlabel('Time step'); ax2.set_ylabel('Absolute error (m)')
ax2.set_title('Estimation Error: Wrong Q Hurts Performance')
ax2.legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f'RMS error (Q too small): {np.sqrt(np.mean((est_small - true_pos_t)**2)):.3f}')
print(f'RMS error (Q too large): {np.sqrt(np.mean((est_large - true_pos_t)**2)):.3f}')
print(f'RMS error (Q correct):   {np.sqrt(np.mean((est_correct - true_pos_t)**2)):.3f}')
print()
print('Q too small: filter becomes overconfident in its prediction and ignores measurements.')
print('Q too large: filter becomes too reactive and chases measurement noise.')
print('Correct Q: filter achieves the optimal balance.')

## Exercises

### Exercise 16.1: Temperature Tracking

A room temperature changes slowly (random walk with small $Q$). A noisy thermometer reads it every second. Implement a 1D Kalman filter to track temperature. The true temperature starts at 20 degrees C and drifts with $\sigma_Q = 0.05$ degrees per step. The thermometer has $\sigma_R = 0.5$ degrees. Run for 200 steps and plot the true temperature, noisy readings, and your filtered estimate.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
T_init      = 20.0    # initial temperature (degrees C)
sigma_Q_t   = 0.05    # process noise std
sigma_R_t   = 0.5     # measurement noise std
n_temp      = 200     # number of steps
# ─────────────────────────────────────────────────────────────────────────────

# YOUR CODE HERE
# 1. Generate ground truth temperature (random walk from T_init)
# 2. Generate noisy thermometer readings
# 3. Implement predict/update loop
# 4. Plot: true temperature, readings, KF estimate with +/- 2 sigma band


### Exercise 16.2: Measurement Rate

How does the measurement frequency affect the filter? Use the 1D setup from Section 16.6. Run three variants:
- Measure every step
- Measure every 5 steps (only predict in between)
- Measure every 10 steps

Plot the covariance $P$ over time for all three. When does the filter's uncertainty grow unacceptably large? What is the relationship between measurement rate and steady state performance?

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
Q_rate      = 0.25    # process noise variance
R_rate      = 1.0     # measurement noise variance
n_rate      = 100     # total steps
rates       = [1, 5, 10]   # measure every N steps
# ─────────────────────────────────────────────────────────────────────────────

# YOUR CODE HERE
# For each rate:
#   1. Run the KF, calling update() only when t % rate == 0
#   2. Store P at each step
# Plot all three P histories on one figure


### Exercise 16.3: Two Sensor Fusion

A robot has two position sensors with different noise levels: $\sigma_1 = 0.5$ m and $\sigma_2 = 2.0$ m. Implement a Kalman filter that fuses both sensors at each time step. Compare the fused result to using each sensor alone.

**Hint:** At each step, you can perform two sequential updates (one per sensor), or a single update with a stacked measurement vector $\mathbf{z} = [z_1, z_2]^T$ and $H = [1; 1]$, $R = \text{diag}(\sigma_1^2, \sigma_2^2)$.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
sigma_s1    = 0.5     # sensor 1 noise std (precise)
sigma_s2    = 2.0     # sensor 2 noise std (noisy)
sigma_Q_f   = 0.3     # process noise std
n_fuse      = 80      # number of steps
# ─────────────────────────────────────────────────────────────────────────────

# YOUR CODE HERE
# 1. Generate ground truth
# 2. Generate two independent measurement streams
# 3. Run three KFs: sensor 1 only, sensor 2 only, both fused
# 4. Plot all three estimates and compare RMS errors


### Exercise 16.4: Steady State Kalman Filter

Since the covariance converges to $P_\infty$, we can precompute the steady state Kalman gain $K_\infty$ and use it for all time steps, saving the matrix computations at each step.

1. Use `scipy.linalg.solve_discrete_are` to compute $P_\infty$ for the 1D system.
2. Compute $K_\infty = P_\infty / (P_\infty + R)$.
3. Run the filter using constant $K_\infty$ (no covariance update needed).
4. Compare the time varying KF and the steady state KF. How quickly does the time varying version "catch up" to steady state performance?

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
Q_ss     = 0.25
R_ss     = 1.0
n_ss     = 100
# ─────────────────────────────────────────────────────────────────────────────

# YOUR CODE HERE
# 1. Compute P_inf using solve_discrete_are
# 2. Compute K_inf
# 3. Run time-varying KF and steady-state KF on the same data
# 4. Plot both estimates and their errors


### Exercise 16.5: Position and Heading Tracking

Track a 2D robot with state $\mathbf{x} = [x, y, \theta]^T$. The robot moves with known velocity commands. Both position $(x, y)$ and heading $\theta$ are measured with noise.

**Note:** This system is only approximately linear (heading wraps around at $\pm\pi$). For small angle changes, the linear KF works; for large turns, you would need an EKF. Assume small angle changes for this exercise.

Set up $F$, $H$, $Q$, $R$ for this system and run the filter. Plot the heading estimate converging to truth.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
dt_h        = 0.1
sigma_pos_h = 0.5     # position measurement noise
sigma_th_h  = 0.1     # heading measurement noise (rad)
sigma_q_h   = 0.05    # process noise std
n_heading   = 100
# ─────────────────────────────────────────────────────────────────────────────

# YOUR CODE HERE
# State: [x, y, theta], Measurement: [x, y, theta]
# F = I (slowly varying), H = I
# Generate truth, measurements, run KF, plot heading convergence


### Exercise 16.6 (Capstone): Multi Rate Vehicle Tracking

A car drives on a highway. Two sensors provide data at different rates:

- **GPS** gives noisy position at **1 Hz** ($\sigma_{\text{GPS}} = 3.0$ m)
- **Accelerometer** gives noisy acceleration at **10 Hz** ($\sigma_{\text{accel}} = 0.5$ m/s$^2$)

The state is $\mathbf{x} = [x, v_x, a_x]^T$ (position, velocity, acceleration).

**Tasks:**
1. Design $F$ for a constant acceleration model at 10 Hz (dt = 0.1 s).
2. Design $H_{\text{GPS}} = [1, 0, 0]$ and $H_{\text{accel}} = [0, 0, 1]$.
3. At each 10 Hz step, predict and update with the accelerometer.
4. Every 10th step (1 Hz), also update with GPS.
5. Plot the position, velocity, and acceleration estimates. Compare the fused result to GPS only and accelerometer only.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
dt_cap         = 0.1      # 10 Hz base rate
gps_rate       = 10       # GPS fires every 10th step (1 Hz)
sigma_gps      = 3.0      # GPS noise std (m)
sigma_accel_c  = 0.5      # accelerometer noise std (m/s^2)
sigma_jerk     = 0.1      # process noise: jerk std
n_cap          = 300      # 30 seconds at 10 Hz
# ─────────────────────────────────────────────────────────────────────────────

# YOUR CODE HERE
# 1. Generate true trajectory: car accelerates, cruises, brakes
# 2. Generate GPS measurements at 1 Hz and accel measurements at 10 Hz
# 3. Implement KF with multi-rate updates
# 4. Plot position, velocity, acceleration estimates
# 5. Compute and compare RMS errors for: GPS only, accel only, fused


## Summary

The Kalman filter is the **optimal estimator** for linear systems with Gaussian noise. Its power comes from a simple two step cycle:

| Step | What it does | Effect on uncertainty |
|---|---|---|
| **Predict** | Propagate the state forward using the motion model | Covariance **grows** (adds $Q$) |
| **Update** | Incorporate a measurement using the Kalman gain | Covariance **shrinks** (information gain) |

Key insights from this chapter:

1. The **Kalman gain** $K$ is the optimal balance between trusting the prediction and trusting the measurement. It equals inverse variance weighting.
2. The covariance **converges to a steady state** determined by the ratio of process noise $Q$ to measurement noise $R$.
3. The filter can **infer unobserved states** (like velocity from position) through the dynamics model.
4. The filter **fails gracefully** when assumptions are violated: wrong noise parameters cause overconfidence or over reactivity, and nonlinear dynamics cause tracking errors during maneuvers.

**Next chapter:** When the world is nonlinear, we linearize the Kalman filter to create the **Extended Kalman Filter (EKF)** and use sigma points for the **Unscented Kalman Filter (UKF)**.